# LUMA - lokalny urok Mistrzostwa Artystycznego

Luma to lokalny jarmark, który rośnie z tygodnia na tydzień, coraz więcej wystawców, więcej gości (miejscowych i turystów), okazjonalne promocje i sezonowość.

# Import bibliotek

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

# Wygenerowanie syntetycznych danych
Ustawienie ziarna RNG


In [2]:
RNG_SEED = 20261026
rng = np.random.default_rng(RNG_SEED)
print("Ustawienie ziarna RNG na ",RNG_SEED)

Ustawienie ziarna RNG na  20261026


# Zdefiniowanie "stałych"


In [3]:
LUMA_FULL = "Lokalny Urok Mistrzostwa Artystycznego"
CATEGORIES = ["Ceramika", "Tekstylia", "Biżuteria", "Rekodzieło", "Przetwory", "Dekoracje", "Stolarka", "Masarstwo", "Alkohole"]

# Przygotowanie bazowych danych syntetycznych 
Katalog kategorii i ich bazowe parametry generatywne(syntetyczne)

In [ ]:
catalog = pd.DataFrame({
    "category": CATEGORIES,
    "demand_loc_base": [120, 100, 130, 80, 160, 150, 70, 90,110], #średni lokalny popyt (szt./tydz.)
    "demand_tur_base": [60, 40, 80, 30, 90, 30, 50, 40, 80], #średni popyt turystyczny (szt./tydz.)
    "price_pln_base": [0, 120, 150, 250, 40, 35, 250, 215, 135] #średnia cena jednostkowa 
})

# Wyliczony "typowy" przychód tygodniowy: (Lokalni + turyści)*cena

In [ ]:
catalog["revenue_avg_pln"] + catalog["demand_lac_base"] + catalog["demand_tur_base"] + catalog["price_pln_base"]

# Przygotowanie funckji pomocniczych do generowania danych tygodniowych

Generowanie popytu lokalnego i turystycznego na podstawie danych bazowych, z losowym szumem (multiplikatywnym) i opcjonalnymi boostami (%)

In [ ]:
def determine_demand(catalog, rng, variance=0.15, boost_loc=0.0, boost_tur=0.0):
    df = catalog[["category", "demand_loc_base", "demand_tur_base"]].copy()
    eps_loc = np.clip(rng.normal(loc=1.0, scale=variance, size=len(df)), 0.5, 1.8)
    eps_tur = np.clip(rng.normal(loc=1.0, scale=variance, size=len(df)), 0.5, 1.8)
    df["demand_loc"] = (df["demand_loc_base"] * (1 + boost_loc)) * eps_loc
    df["demand_tur"] = (df["demand_tur_base"] * (1 + boost_tur)) * eps_tur 
    df["demand_loc"] = np.maximum(df["demand_loc"].round().astype(int), 0)
    df["demand_tur"] = np.maximum(df["demand_tur"].round().astype(int), 0)
    return df[["category", "demand_loc", "demand_tur"]]

Wyliczanie przychodu lokalnego, turystycznego i łącznego

In [ ]:
def count_revenue(catalog, demand_df):
    df = demand_df.merge(catalog[["category", "price_pln_base"]], on="category", how="left")
    df["revenue_loc"] = df["demand_loc"] * df["price_pln_base"]
    df["revenue_tur"] = df["demand_tur"] * df["price_pln_base"]
    df["revenue_pln"] = df["revenue_loc"] + df["revenue_tur"]
    return df[["category", "revenue_loc", "revenue_tur", "revenue_pln"]]

Generowanie danych tygodniowych

In [ ]:
def generate_weekly_data(catalog, rng, variance=0.15, boost_loc=0.0, boost_tur=0.0):
    demand_df = determine_demand(catalog, rng, variance=variance, boost_loc=boost_loc, boost_tur=boost_tur)
    revenue_df = count_revenue(catalog, demand_df)
    return revenue_df

## Wygenerowanie danych tygodniowych

In [ ]:
t1 = generate_weekly_data(catalog, rng)
t1 = t1.merge(catalog[["category", "price_pln_base"]], on="category", how="left")
print("Tydzień 1 - przychody per kategoria (PLN)")
display(t1)
print("Łączny przychód w tygodniu 1 (PLN): ", t1["revenue_pln"].sum())

Tydzień 1 - przychody per kategoria (PLN)

# Analiza i wizualizacja danych
Podstawowe wykresy

In [ ]:
t1.plot.bar()

In [ ]:
t1[["revenue_loc", "revenue_tur"]].plot.bar()

In [ ]:
t1[["revenue_loc", "revenue_tur"]].plot.barh(stacked=True)

In [ ]:
hoights = t1["revenue_pln"]
plt.bar(range(len(hoights)), hoights)

Jawne tworzenie i wyświetlanie wykresu

In [ ]:
hoights = t1["revenue_pln"]
categories = t1["category"]

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(categories, hoights)
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(categories, hoights)
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.show()

Etykiety wartości i formatowanie liczb